In [ ]:
#import libbary
import numpy as np
import pandas as pd

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import mpl_toolkits.mplot3d as Axes3D #untuk 3d plot
#agar plot tampil rapi
plt.rcParams['figure.figsize'] = (8, 6)


In [ ]:
#load dataset wine
wine = load_wine()

df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target
df.head()

In [ ]:
#informasi umum dataset
df.info()

In [ ]:
#stastistik dekskriptif
df.describe()

In [ ]:
#cek missing value
df.isnull().sum()

In [ ]:
#cek data duplikat
df.duplicated().sum()

In [ ]:
#distribusi kelas target
print("nama kelas : ", wine.target_names)
df['target'].value_counts()

In [ ]:
#pemisahan fitur dan label
X = wine.data   #bisa juga : df[wine.feature_names]
y = wine.target  #df['target']

print('Shape of X : ', X.shape)
print('Shape of y : ', y.shape)

In [ ]:
#pembagian data latih dan uji
#train test split (80% train,20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('Shape of X_train : ', X_train.shape)
print('Shape of y_train : ', y_train.shape)

In [ ]:
#standardisasi (mean=0,std=1)
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train[:5]

In [ ]:
# Model SVM tanpa PCA
svm_no_pca = SVC(kernel='rbf', gamma='scale', random_state=42)
svm_no_pca.fit(X_train, y_train)

# Prediksi dan evaluasi
y_pred_no_pca = svm_no_pca.predict(X_test)

acc_no_pca = accuracy_score(y_test, y_pred_no_pca)
print("Akurasi SVM tanpa PCA:", acc_no_pca)

print("\nClassification Report (tanpa PCA):")
print(classification_report(y_test, y_pred_no_pca, target_names=wine.target_names))

In [ ]:
#pca dengan 3 komponen utama
pca = PCA(n_components=3)

X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print("shape X_train_pca:", X_train_pca.shape)
print("shape X_test_pca:", X_test_pca.shape)

In [ ]:
#variasi yang di jelaskan oleh tiap komponen
explaine_var = pca.explained_variance_ratio_
print("Explained variance ratio tiap komponen: ", explaine_var)
print("total variasi yang di jelaskan 3 komponen pertama: ", explaine_var.sum())

In [ ]:
plt.bar([1,2,3], explaine_var)
plt.xlabel('Komponen utama')
plt.ylabel('Explained variance ration')
plt.title('variasi yang di jelaskan 3 komponen PCA')
plt.show()

In [ ]:
# Model SVM dengan PCA
svm_pca = SVC(kernel='rbf', gamma='scale', random_state=42)
svm_pca.fit(X_train_pca, y_train)

# Prediksi dan evaluasi
y_pred_pca = svm_pca.predict(X_test_pca)

acc_pca = accuracy_score(y_test, y_pred_pca)
print("Akurasi SVM dengan PCA (3 komponen):", acc_pca)

print("\nClassification Report (dengan PCA):")
print(classification_report(y_test, y_pred_pca, target_names=wine.target_names))

In [ ]:
# Visualisasi 3D PCA (menggunakan data train)
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(
    X_train_pca[:, 0],
    X_train_pca[:, 1],
    X_train_pca[:, 2],
    c=y_train,
    s=60
)

ax.set_title('Visualisasi PCA (3 Komponen) - Dataset Wine')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')

# Tambahkan legend berdasarkan kelas
legend1 = ax.legend(
    *scatter.legend_elements(),
    title="Kelas"
)
ax.add_artist(legend1)

plt.show()

In [ ]:
# Membandingkan hasil model tanpa dan dengan PCA
comparison = pd.DataFrame({
    'Model': ['SVM Tanpa PCA', 'SVM Dengan PCA (3 Komponen)'],
    'Jumlah Fitur': [X_train.shape[1], X_train_pca.shape[1]],
    'Akurasi': [acc_no_pca, acc_pca],
    'Variansi Total PCA': [None, explaine_var.sum()]
})

comparison

In [ ]:
plt.figure(figsize=(8, 6))
plt.bar(['TANPA PCA', 'DENGAN PCA (3 KOMPONEN)'], [acc_no_pca, acc_pca])
plt.title('Perbandingan Akurasi Model SVM')
plt.ylabel('Akurasi')
plt.ylim(0,1)
for i, v in enumerate([acc_no_pca, acc_pca]):
  plt.text(i, v + 0.01,f'{v:.2f}', ha='center')
plt.show()

 Kesimpulan Akhir Praktikum PCA
Pada praktikum ini telah dilakukan penerapan Principal Component Analysis (PCA) sebagai salah satu
teknik reduksi dimensi untuk menyederhanakan dataset tanpa mengurangi informasi penting secara
signifikan. PCA diimplementasikan pada dataset Sirup, yang awalnya memiliki 13 fitur numerik dan 3
kelas target (jenis Sirup berbeda).
Dari seluruh tahapan yang dilakukan, diperoleh beberapa kesimpulan utama sebagai berikut:
1. Proses reduksi dimensi dengan PCA berhasil mengubah 13 fitur asli menjadi 3 komponen utama
(PC1, PC2, dan PC3). Ketiga komponen tersebut mampu menjelaskan sekitar 66.08% variansi total
dari data asli, yang berarti sebagian besar informasi masih dapat dipertahankan.
2. Model SVM tanpa PCA dan model SVM dengan PCA (3 komponen) menunjukkan akurasi yang
sama tinggi, yaitu 97.22%. Hal ini membuktikan bahwa penerapan PCA tidak menurunkan
performa klasifikasi meskipun jumlah fitur berkurang drastis
3. Visualisasi 3D PCA menunjukkan bahwa data dari tiga kelas (class_0, class_1, dan class_2) dapat
terpisah dengan cukup baik pada ruang tiga dimensi hasil transformasi PCA, menandakan bahwa
komponen utama berhasil menangkap struktur data yang relevan.
4. Dengan menerapkan PCA, model menjadi lebih efisien secara komputasi, karena bekerja dengan
jumlah fitur yang lebih sedikit, namun tetap mempertahankan kinerja prediksi yang sangat baik.
5. PCA juga membantu mengurangi risiko overfitting, terutama pada dataset dengan dimensi tinggi,
karena menghilangkan korelasi antar fitur dan menyederhanakan representasi data.